In [ ]:
import sys
import math
from random import randint
import pandas as pd
from PySide6 import QtGui, QtWidgets, QtCore
from PySide6.QtCore import QThread, Signal, QMutex, QMutexLocker
from PySide6.QtWidgets import QApplication, QMainWindow, QFileDialog, QDialog, QLabel, QVBoxLayout, QWidget, QGraphicsPathItem, QMessageBox, QErrorMessage
from PySide6.QtCore import QTimer 
import pyqtgraph as pg
from pyqtgraph.Qt import QtCore  
import numpy as np
import serial
import serial.tools.list_ports
import time
from data_collection_ui import Ui_MainWindow
import os
import csv
from datetime import datetime
import serial.tools.list_ports

Serial Reader Thread

In [ ]:
class SerialReaderThread(QThread):
    # data_received = Signal(list, list)  # Signal for right and left forces (not used)
    connection = Signal(bool) # Signal to emit to main window to indicate serial connection
    error_message_signal = Signal(str, str)  # Signal for error message


    def __init__(self):
        super().__init__()

        self.baudrate = 115200
        self.running = False
        self.serial = None # initialize as none
        self.com_port = None

        global file_path
        file_path = None

        # Calibration factors for each channel
        self.calibration_factors = {
            '1A': 0.05053931524,  
            '1B': 0.05109740826,
            '1C': 0.05108326319,
            '1D': 0.05021199964,
            '2A': 0.04986771817,
            '2B': 0.05140285445,
            '2C': 0.05292458815,
            '2D': 0.05131591832}
        # Initialize data
        self.raw_ADC_vals = {}
        self.channel_force_vals = {}
        self.tared_channel_values = {}
        self.channel_tares = {'1A':0, '1B':0, '1C':0, '1D':0,
        '2A':0, '2B':0, '2C':0, '2D':0}
        self.force_left = 0
        self.force_right = 0
        self.saving_data = False
        self.last_write = None
        self.new_tares = False

        global usb_port
        usb_port = ""
        self.no_data_start_time = None  # Track the start time when no data is received
        self.error_threshold = 1  # Time threshold in seconds
   

    def autodetect_com_port(self):
        """Auto-detect the Raspberry Pi Pico's COM port on macOS."""
        global usb_port
        ports = serial.tools.list_ports.comports()

        for port in ports:
            if "usb" in port.device.lower() or "usb" in port.description.lower() or "Pico" in port.description:
                self.com_port = port.device
                usb_port = str(port.device)
                return True
    
        print("No Raspberry Pi Pico found.")
        return False


    def run(self):
        """Thread execution starts here."""
        if not self.autodetect_com_port():
            return

        try:
            self.serial = serial.Serial(self.com_port, self.baudrate, timeout=1)
            self.running = True

            while self.running:
                try:
                    # Check if we have any data to read, and make sure the connection is valid
                    if self.serial.in_waiting:
                        line = self.serial.readline().decode('utf-8').strip()
                        if line:
                            channel, value = line.split(':')
                            channel = channel.strip()
                            value = int(value.strip())
                            self.update_channel_value(channel, value)
                            self.connection.emit(True)
                            self.no_data_start_time = None
                    else:
                        # If there's no data but the serial connection is active, check if it's open
                        if not self.serial.is_open:
                            raise serial.SerialException("Connection lost during read.")
                        else:
                            if self.no_data_start_time is None:
                                self.no_data_start_time = time.time()
                            if time.time() - self.no_data_start_time >= self.error_threshold:
                                self.error_message_signal.emit("Serial Error", "Not receiving data from pico. Unplug and replug the USB code, and restart the GUI program.")
                                break
                except serial.SerialException as e:
                    # Handle serial connection loss or any other serial error
                    print(f"Serial Error: {e}")
                    self.error_message_signal.emit("Serial Error", f"Connection lost: {e}")
                    self.connection.emit(False)
                    self.running = False
                    break
                except Exception as e:
                    # Catch any other exceptions (e.g., parsing errors)
                    print(f"Unexpected error: {e}")
                    self.error_message_signal.emit("Unexpected Error", str(e))
                    self.connection.emit(False)
                    self.running = False
                    break

        finally:
            # Ensure the serial connection is properly closed
            if self.serial and self.serial.is_open:
                self.connection.emit(False)
                self.serial.close()
                print("Serial connection closed safely.")

    def update_channel_value(self, channel, value):
        """Update values and compute force sums."""
        if channel in self.calibration_factors:
            self.raw_ADC_vals[channel] = round(value,6)
            calibrated_value = value * self.calibration_factors[channel]
            self.channel_force_vals[channel] = round(calibrated_value,6)
            self.tared_channel_values[channel] = round(calibrated_value - self.channel_tares[channel],6)
            self.compute_sums()

    def compute_sums(self):
        """Compute the sum of values for left and right force sensors."""
        self.force_right = round(sum(value for channel, value in self.tared_channel_values.items() if channel[0] == '1'), 4)
        self.force_left = round(sum(value for channel, value in self.tared_channel_values.items() if channel[0] == '2'), 4)
        if self.saving_data:
            if (self.last_write == None) or (time.time() - self.last_write >= 0.001):
                self.write_csv_row()
    
    def tare(self):
        self.channel_tares = self.channel_force_vals.copy()  # Ensure independent copy
        self.channel_tares = {k: round(v, 6) for k, v in self.channel_tares.items()}  # Reduce floating-point drift
        self.new_tares = True

    def measure_now(self):
        self.open_csv()
        self.write_csv_row()

    def start_data_collection(self):
        self.open_csv()
        self.saving_data = True

    def stop_data_collection(self):
        self.saving_data = False
    
    def open_csv(self):
        # checks if there's a file path, opens csv and writes header
        global file_path
        if file_path == None:
            self.show_error_message("Error", "No file path selected.")

            print("no file path selected")
        else:
            print(file_path)
            """
            Write a single row of data to a CSV file. If the file doesn't exist, it creates one.
            
            :param file_name: Full path of the CSV file.
            :param data_row: List of values to write as a row.
            """
            file_exists = os.path.isfile(file_path)  # Check if the file exists

            try:
                with open(file_path, mode='a', newline='') as file:  # Open in append mode
                    writer = csv.writer(file)

                    # Write the header only if the file doesn't exist
                    if not file_exists:
                        writer.writerow(["Timestamp", "Force Left (lbs)", "Force Right (lbs)", 
                                        "Force RA (lbs)", "Force RB (lbs)", "Force RC (lbs)", "Force RD (lbs)", 
                                        "Force LA (lbs)", "Force LB (lbs)", "Force LC (lbs)", "Force LD (lbs)",
                                        "Raw RA (ADC)", "Raw RB (ADC)", "Raw RC (ADC)", "Raw RD (ADC)",
                                        "Raw LA (ADC)", "Raw LB (ADC)", "Raw LC (ADC)", "Raw LD (ADC)",
                                        "Tare RA (ADC)", "Tare RB (ADC)", "Tare RC (ADC)", "Tare RD (ADC)",
                                        "Tare LA (ADC)", "Tare LB (ADC)", "Tare LC (ADC)", "Tare LD (ADC)"])
            except Exception as e:
                print(f"Data Saving Error: {e}")
                self.error_message_signal.emit("Data Saving Error", str(e))

    def write_csv_row(self):

        with open(file_path, mode='a', newline='') as file:  # Open in append mode
            writer = csv.writer(file)
            row = [str(datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]), self.force_left, self.force_right]

            # print statement can be used to monitor connection (optional)
            # print(self.tared_channel_values["1A"])
            
            for channel in self.calibration_factors.keys():
                row.append(self.tared_channel_values[channel])
            for channel in self.calibration_factors.keys():
                row.append(self.raw_ADC_vals[channel])
            if self.new_tares:
                for channel in self.calibration_factors.keys():
                    row.append(self.channel_tares[channel])
                self.new_tares = False
            writer.writerow(row)# Write the data 
            self.last_write = time.time()

    def stop(self):
        """Stop the thread safely."""
        self.running = False
        if self.serial and self.serial.is_open:
            self.serial.close()


Main Window

In [ ]:
class MainWindow(QMainWindow):
    def __init__(self, parent=None):
        super().__init__()
        
        # Receive signal from SerialReaderThread
        self.serial_thread = SerialReaderThread()
        #self.serial_thread.data_received.connect(self.update_function)  #not used
        self.serial_thread.error_message_signal.connect(self.show_error_message)
        self.serial_thread.connection.connect(self.connection_label)
        self.serial_thread.start()

        self.left_tare = 0
        self.right_tare = 0

        # Set up UI
        self.ui = Ui_MainWindow()
        self.ui.setupUi(self)

        # Configure file dialog with browse button
        self.ui.browse_btn.clicked.connect(self.open_file_dialog)

        # Set mode to continuous or on demand
         # Create button group to allow only one selection
        self.ui.button_group = QtWidgets.QButtonGroup(self.ui.centralwidget)
        self.ui.button_group.addButton(self.ui.continuous_btn)
        self.ui.button_group.addButton(self.ui.ondemand_btn)
        self.ui.continuous_btn.setChecked(True) #set continuous as default
        self.mode = "Continuous" #default mode
        self.ui.measure_btn.setEnabled(False) #default mode
        self.ui.continuous_btn.toggled.connect(self.set_mode)

        self.ui.stop_btn.setEnabled(False)
        self.ui.start_btn.clicked.connect(self.start_data_collection)
        self.ui.stop_btn.clicked.connect(self.stop_data_collection)
        self.ui.tare_btn.clicked.connect(self.tare)
        self.ui.measure_btn.clicked.connect(self.measure_now)

        self.ui.connection_label.setText("Not connected")

        self.ui.line_edit.textChanged.connect(self.update_file_path)  # Connect text change event

        self.timer = QTimer(self)
        self.timer.timeout.connect(self.update_timer)

        global file_path
        file_path = None

    def update_function(self, right_list, left_list):
        self.raw_force_left = left_list[-1]
        self.raw_force_right = right_list[-1]

    def open_file_dialog(self):
        # Open File Dialog to select folder and enter a filename
        global file_path
        file_path, _ = QFileDialog.getSaveFileName(self, "Save CSV File", "", "CSV Files (*.csv);;All Files (*)")

        if file_path:  # If the user selects a path
            if not file_path.endswith(".csv"):  # Ensure it has a .csv extension
                file_path += ".csv"
            self.ui.line_edit.setText(file_path) # Display selected path in QLineEdit

    def update_file_path(self):
        # Update file_path with the content of the QLineEdit
        global file_path
        file_path = self.ui.line_edit.text()

    def set_mode(self):
        if self.ui.continuous_btn.isChecked():
            self.mode = "Continuous"
            self.ui.start_btn.setEnabled(True)
            self.ui.measure_btn.setEnabled(False)
        elif self.ui.ondemand_btn.isChecked():
            self.mode = "On demand"
            self.ui.measure_btn.setEnabled(True)
            self.ui.start_btn.setEnabled(False)
            self.ui.stop_btn.setEnabled(False)

    def tare(self):
        self.ui.tare_btn.setStyleSheet("background-color: green; color: white;")
        self.serial_thread.tare()

    def measure_now(self):
        if file_path == None:
            self.show_error_message("Error", "No file path selected.")
        else:
            self.serial_thread.measure_now()

    def update_timer(self):
        self.time_elapsed = time.time() - self.start_time
        self.time_elapsed = time.strftime("%H:%M:%S", time.gmtime(self.time_elapsed))
        self.ui.time_label.setText(str(self.time_elapsed))

    def start_data_collection(self):
        if file_path == None:
            self.show_error_message("Error", "No file path selected.")
        else:
            self.ui.start_btn.setEnabled(False)
            self.ui.stop_btn.setEnabled(True)
            self.start_time = time.time()
            self.timer.start(500) # Update every second 40 ms = 25 Hz
            self.serial_thread.start_data_collection()

    def stop_data_collection(self):
        self.timer.stop()
        self.ui.stop_btn.setEnabled(False)
        self.ui.start_btn.setEnabled(True)
        self.serial_thread.stop_data_collection()

    def connection_label(self, connection_status):
        if connection_status:
            self.ui.connection_label.setText(f'Connected on {usb_port}')
        else:
            self.ui.connection_label.setText("CONNECTION LOST")
            self.ui.connection_label.setStyleSheet("background-color: red")

    def show_error_message(self, title, message):
        """
        Display an error message using QMessageBox.
        
        :param title: The title of the error message box.
        :param message: The message to be displayed.
        """
        msg_box = QMessageBox()
        msg_box.setIcon(QMessageBox.Critical)
        msg_box.setWindowTitle(title)
        msg_box.setText(message)
        msg_box.exec_()


In [ ]:
if __name__ == "__main__":
    app = QApplication(sys.argv)
    widget = MainWindow()
    widget.show()
    sys.exit(app.exec())